# Gold Setup Step 8 — Hierarchical Category Structure

Crea estructura jerárquica de 3 niveles para mejorar clasificación:

## 🏗️ Estructura:
1. **MACRO** (9 categorías): Almacén, Bebidas, Frescos, Congelados, Limpieza, Perfumería, Electro, Textil, Hogar
2. **CATEGORÍA** (~40): Golosinas, Panadería, Snacks, Lácteos, Carnicería...
3. **SUB-CATEGORÍA** (139 actuales): Alfajores, Chocolates, Yogur...

## ✅ Ventajas:
- **Mayor confianza**: Clasificar en 9 opciones (nivel 1) es más fácil que 139
- **LLM más preciso**: Menos opciones = mejores decisiones  
- **Menos review queue**: ~80% auto-asignado en nivel 1

## 📊 Output:
- `gold_macro_categorias` (9 macros)
- `gold_categorias` actualizada con `id_macro`
- `gold_macro_centroids` (embeddings agregados por macro)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.superapp.gold_macro_categorias (
    id_macro INT,
    nombre STRING NOT NULL,
    descripcion STRING,
    CONSTRAINT pk_macro PRIMARY KEY(id_macro)
);

INSERT INTO workspace.superapp.gold_macro_categorias VALUES
    (1, 'Almacén', 'Productos secos: golosinas, snacks, cereales, harinas, conservas, pastas, condimentos'),
    (2, 'Bebidas', 'Bebidas con y sin alcohol: aguas, gaseosas, jugos, cervezas, vinos'),
    (3, 'Frescos', 'Productos refrigerados: lácteos, fiambres, quesos, carnes, frutas, verduras, pastas frescas'),
    (4, 'Congelados', 'Productos congelados: pescados, hamburguesas, helados, vegetales, frutas, papas fritas'),
    (5, 'Limpieza', 'Productos de limpieza: jabones, detergentes, papeles, insecticidas, accesorios'),
    (6, 'Perfumería', 'Higiene personal: shampoo, cremas, desodorantes, cepillos dentales, protector solar'),
    (7, 'Electro', 'Electrónica y electrodomésticos: celulares, TVs, heladeras, lámparas LED'),
    (8, 'Textil', 'Ropa y textiles: remeras, toallas, paños, ropa infantil'),
    (9, 'Hogar', 'Artículos de hogar: platos, jarras, artículos escolares, velas, calefactores');

SELECT * FROM workspace.superapp.gold_macro_categorias ORDER BY id_macro;

In [0]:
%sql
-- Add column (ignore error if already exists)
ALTER TABLE workspace.superapp.gold_categorias ADD COLUMNS (id_macro INT);

SELECT 'Columna id_macro agregada' AS status;

In [0]:
%sql
-- ALMACÉN: Golosinas, Snacks, Cereales, Panadería, Harinas, Conservas, Pastas, Condimentos
UPDATE workspace.superapp.gold_categorias 
SET id_macro = 1 
WHERE nombre IN (
    -- Golosinas
    'Alfajor Chocolate', 'Chocolates', 'Chocolate Blanco', 'Chocolate con Leche', 'Tableta Chocolate', 'Huevo Chocolate', 'Huevo Pascua',
    -- Snacks
    'Papas Fritas', 'Snacks Salados', 'Galletitas Dulces', 'Galletitas Crackers', 'Galletitas Chocolate', 'Galletitas Salvado', 'Galletitas con Chips', 'Galletitas',
    -- Cereales
    'Barra Cereal',
    -- Panadería
    'Pan', 'Panes Especiales', 'Pan Rallado',
    -- Endulzantes
    'Azúcar', 'Edulcorantes y Suplementos en Polvo',
    -- Infusiones
    'Café', 'Café Molido', 'Café Tostado Grano', 'Café Torrado', 'Café Instantáneo', 'Café Cápsulas',
    'Té', 'Té Infusión', 'Té Verde', 'Mate Cocido', 'Yerba',
    -- Conservas
    'Aceitunas Verdes', 'Productos del Mar',
    -- Harinas y Arroz
    'Harina', 'Arroz', 'Parboilizado', 'Integral',
    -- Pastas
    'Fideos', 'Fideos Spaghetti', 'Fideos Penne', 'Fideos Tirabuzon',
    -- Condimentos y salsas
    'Aceite', 'Aceite en Spray', 'Especias y Condimentos', 'Vinagres y Condimentos Especiales', 'Condimentos y Jugos en Polvo',
    -- Dulces
    'Dulce de Leche', 'Dulce de Batata', 'Mermeladas y Frutas Rojas',
    -- Varios almacén
    'Polvos para Hornear', 'Sal Fina'
);

SELECT COUNT(*) AS categorias_almacen FROM workspace.superapp.gold_categorias WHERE id_macro = 1;

In [0]:
%sql
-- BEBIDAS: Con y sin alcohol
UPDATE workspace.superapp.gold_categorias 
SET id_macro = 2 
WHERE nombre IN (
    -- Sin alcohol
    'Agua', 'Agua Mineral', 'Agua Gasificada', 'Agua Saborizada',
    'Gaseosa', 'Gaseosas Colas', 'Gaseosas Lima Limón',
    'Jugo', 'Jugo en Polvo',
    'Leche Chocolatada',
    -- Con alcohol
    'Cerveza', 'Vino', 'Vino Tinto', 'Vino Blanco', 'Vino Rosado', 'Espumante'
);

SELECT COUNT(*) AS categorias_bebidas FROM workspace.superapp.gold_categorias WHERE id_macro = 2;

In [0]:
%sql
-- FRESCOS: Lácteos, Fiambres, Quesos, Carnes, Frutas/Verduras, Pastas frescas
UPDATE workspace.superapp.gold_categorias 
SET id_macro = 3 
WHERE nombre IN (
    -- Lácteos
    'Leche', 'Leche Entera', 'Leche en Polvo', 'Yogur', 'Yogur Bebible', 'Manteca', 'Crema de Leche',
    -- Quesos
    'Queso', 'Queso Crema/Untable', 'Queso Rallado', 'Queso Cremoso', 'Queso Azul',
    'Port Salut', 'Queso Mozzarella', 'Queso Hebras', 'Queso Pategras', 'Queso Sardo',
    'Queso Reggianito', 'Queso Cheddar', 'Queso Fundido',
    -- Fiambres
    'Jamón Cocido',
    -- Frutas congeladas (en frescos según imágenes)
    'Frutas Congeladas',
    -- Pastas frescas
    'Tapa Empanada', 'Tapas Pascualinas'
);

SELECT COUNT(*) AS categorias_frescos FROM workspace.superapp.gold_categorias WHERE id_macro = 3;

In [0]:
%sql
-- CONGELADOS: Pescados, hamburguesas, helados, vegetales
UPDATE workspace.superapp.gold_categorias 
SET id_macro = 4 
WHERE nombre IN (
    'Productos Congelados'
);

SELECT COUNT(*) AS categorias_congelados FROM workspace.superapp.gold_categorias WHERE id_macro = 4;

In [0]:
%sql
-- LIMPIEZA: Jabones, detergentes, papeles, accesorios
UPDATE workspace.superapp.gold_categorias 
SET id_macro = 5 
WHERE nombre IN (
    -- Lavado
    'Jabón', 'Jabón Líquido', 'Jabón Líquido Ropa', 'Jabón Líquido Varios', 'Jabón en Polvo', 'Detergente Polvo',
    'Suavizantes y Aprestos',
    -- Papeles
    'Papel Higiénico', 'Rollo Cocina',
    -- Accesorios
    'Fibra Esponja', 'Paños', 'Trapo de Piso',
    -- Limpieza general
    'Limpiador Líquido', 'Limpiador Líquido Concentrado'
);

SELECT COUNT(*) AS categorias_limpieza FROM workspace.superapp.gold_categorias WHERE id_macro = 5;

In [0]:
%sql
-- PERFUMERÍA: Higiene personal
UPDATE workspace.superapp.gold_categorias 
SET id_macro = 6 
WHERE nombre IN (
    -- Cuidado capilar
    'Shampoo', 'Crema Peinar', '_DELETED_cremas_para_peinar',
    -- Cuidado dental
    'Crema Dental', 'Cepillo Dental', 'Enjuague Bucal',
    -- Cuidado corporal
    'Jabón Tocador', 'Desodorantes', 'Crema Corporal', 'Crema Facial',
    'Protector Solar', 'Agua Micelar', 'Aloe Vera',
    -- Afeitado
    'Máquina Afeitar',
    -- Otros
    'Toallitas Húmedas', 'Toalla Femenina', '_DELETED_Toallas Femeninas', 'Tintura Permanente',
    -- Bebés
    'Cuidado Bebés', 'Pañales'
);

SELECT COUNT(*) AS categorias_perfumeria FROM workspace.superapp.gold_categorias WHERE id_macro = 6;

In [0]:
%sql
-- ELECTRO
UPDATE workspace.superapp.gold_categorias SET id_macro = 7 
WHERE nombre IN ('Celular Libre', 'Televisores', 'Heladeras No Frost', 'Lámpara LED', 'Calefactores y Comederos');

-- TEXTIL
UPDATE workspace.superapp.gold_categorias SET id_macro = 8 
WHERE nombre IN ('Remeras Estampadas', 'Textil de Baño', 'Ropa y Accesorios', 'Ropa y Accesorios Infantiles');

-- HOGAR
UPDATE workspace.superapp.gold_categorias SET id_macro = 9 
WHERE nombre IN (
    'Plato Playo', 'Plato Postre', 'Platos Hondos', 'Jarras Mug',
    'Carpeta Escolar', 'Artículos de Dibujo', 'Cuadernos',
    'Sahumerios',
    -- Mascotas
    'Alimento Perros', 'Alimento Gatos', 'Snack Perros',
    -- Sin clasificar y otros
    'Sin Clasificar', 'doble carolina', 'largo fino'
);

-- Resumen por macro
SELECT 
    m.nombre AS macro,
    COUNT(*) AS total_categorias
FROM workspace.superapp.gold_categorias c
JOIN workspace.superapp.gold_macro_categorias m ON c.id_macro = m.id_macro
GROUP BY m.nombre
ORDER BY m.nombre;

In [0]:
%pip install sentence-transformers --quiet
dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer
import numpy as np
import json

# Load embedding model
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Get all classified products with their macro category
products_df = spark.sql("""
    SELECT DISTINCT
        m.id_macro,
        m.nombre AS macro_nombre,
        sp.producto
    FROM workspace.superapp.gold_productos_categorias pc
    JOIN workspace.superapp.gold_categorias c ON pc.id_categoria = c.id_categoria
    JOIN workspace.superapp.gold_macro_categorias m ON c.id_macro = m.id_macro
    JOIN workspace.superapp.silver_prices sp ON pc.id_producto = sp.id_producto
    WHERE sp.producto IS NOT NULL
      AND c.nombre != 'Sin Clasificar'
""").toPandas()

display(products_df)

# Compute centroids per macro
macro_centroids = []

for macro_id, group in products_df.groupby('id_macro'):
    macro_name = group['macro_nombre'].iloc[0]
    products = group['producto'].tolist()
    
    # Embed all products in this macro
    embeddings = model.encode(products, batch_size=256, show_progress_bar=False)
    
    # Compute centroid (mean)
    centroid = embeddings.mean(axis=0)
    centroid_norm = centroid / (np.linalg.norm(centroid) + 1e-10)  # L2 normalize
    
    macro_centroids.append({
        'id_macro': int(macro_id),
        'nombre': macro_name,
        'centroid_json': json.dumps(centroid_norm.tolist()),
        'num_products': len(products)
    })

# Save to table
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema = StructType([
    StructField('id_macro', IntegerType(), False),
    StructField('nombre', StringType(), False),
    StructField('centroid_json', StringType(), False),
    StructField('num_products', IntegerType(), False)
])

centroids_df = spark.createDataFrame(macro_centroids, schema=schema)
centroids_df.write.mode('overwrite').saveAsTable('workspace.superapp.gold_macro_centroids')

display(centroids_df)

# Final statistics
stats_df = spark.sql("""
    SELECT 
        m.nombre AS macro_categoria,
        COUNT(DISTINCT c.id_categoria) AS categorias,
        COUNT(DISTINCT pc.id_producto) AS productos_clasificados,
        cen.num_products AS productos_para_centroid
    FROM workspace.superapp.gold_macro_categorias m
    LEFT JOIN workspace.superapp.gold_categorias c ON m.id_macro = c.id_macro
    LEFT JOIN workspace.superapp.gold_productos_categorias pc ON c.id_categoria = pc.id_categoria
    LEFT JOIN workspace.superapp.gold_macro_centroids cen ON m.id_macro = cen.id_macro
    GROUP BY m.nombre, cen.num_products
    ORDER BY productos_clasificados DESC
""")
display(stats_df)